In [2]:
# ================================
#INSTALL & IMPORT

!pip install transformers -q

import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

# ================================
#LOAD DATA
# ================================
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# ================================
#PREPROCESSING
# ================================
train_df['text'] = "Title: " + train_df['Review_Title'].astype(str) + " Review: " + train_df['Review'].astype(str)
test_df['text'] = "Title: " + test_df['Review_Title'].astype(str) + " Review: " + test_df['Review'].astype(str)

train_df = train_df.dropna(subset=['text', 'Rating'])
test_df = test_df.fillna('')

# ================================
#TRAIN-VALIDATION SPLIT
# ================================
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text'],
    train_df['Rating'],
    test_size=0.2,
    stratify=train_df['Rating'],
    random_state=42
)

#IMPORTANT FIX (avoid bugs)
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_val = y_val.reset_index(drop=True)

# ================================
#TOKENIZATION
# ================================
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=256)
test_encodings = tokenizer(list(test_df['text']), truncation=True, padding=True, max_length=256)

# ================================
#DATASET CLASS
# ================================
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = Dataset(train_encodings, list(y_train))
val_dataset = Dataset(val_encodings, list(y_val))
test_dataset = Dataset(test_encodings)

# ================================
#MODEL
# ================================
model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

# ================================
#TRAINING SETTINGS
# ================================
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=100,
)

# ================================
#TRAINER
# ================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# ================================
#TRAIN MODEL
# ================================
trainer.train()

# ================================
#VALIDATION PREDICTIONS
# ================================
val_preds = trainer.predict(val_dataset)
logits = val_preds.predictions
probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()

# ================================
#THRESHOLD TUNING (F1 OPTIMIZATION)
# ================================
best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.2, 0.8, 0.01):
    preds = (probs[:,1] > t).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best F1 Score:", best_f1)

# ================================
#TEST PREDICTIONS
# ================================
test_preds = trainer.predict(test_dataset)
test_logits = test_preds.predictions
test_probs = torch.nn.functional.softmax(torch.tensor(test_logits), dim=1).numpy()

# ================================
#FINAL SUBMISSION
# ================================
final_preds = (test_probs[:,1] > best_thresh).astype(int)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Rating": final_preds
})

submission.to_csv("submission.csv", index=False)

print("Submission file created successfully!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.042994,0.051287
2,0.019442,0.047655
3,0.013160,0.059027


Best Threshold: 0.2
Best F1 Score: 0.9945054945054945


🚀 Submission file created successfully!


In [3]:
train_df = train_df.drop_duplicates(subset=['Review_Title', 'Review'])

In [4]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text'],
    train_df['Rating'],
    test_size=0.2,
    stratify=train_df['Rating'],
    random_state=42
)

In [5]:
train_texts = set(X_train)
val_texts = set(X_val)

overlap = len(train_texts.intersection(val_texts))
print("Overlap:", overlap)

Overlap: 0


In [7]:
val_encodings = tokenizer(list(X_val), truncation=True, padding=True, max_length=256)
val_dataset = Dataset(val_encodings, list(y_val))

val_preds = trainer.predict(val_dataset)
logits = val_preds.predictions
probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1).numpy()

best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.2, 0.8, 0.01):
    preds = (probs[:,1] > t).astype(int)
    f1 = f1_score(y_val, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best Threshold:", best_thresh)
print("Best F1 Score:", best_f1)

# ================================
#TEST PREDICTIONS
# ================================
test_preds = trainer.predict(test_dataset)
test_logits = test_preds.predictions
test_probs = torch.nn.functional.softmax(torch.tensor(test_logits), dim=1).numpy()

# ================================
#FINAL SUBMISSION
# ================================
final_preds = (test_probs[:,1] > best_thresh).astype(int)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Rating": final_preds
})

submission.to_csv("submission.csv", index=False)

print(" Submission file created successfully!")

Best Threshold: 0.2
Best F1 Score: 0.9961378490790256


 Submission file created successfully!


In [9]:
from transformers import pipeline

preds = (probs[:,1] > best_thresh).astype(int)

print(preds[:20])
print(list(y_val)[:20])

[1 1 1 1 1 0 1 1 0 1 1 1 0 1 1 0 1 1 1 0]
[1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0]


In [11]:
trainer.predict(test_dataset)

PredictionOutput(predictions=array([[-4.227596 ,  4.7187963],
       [-4.2449183,  4.7311435],
       [-4.2381897,  4.726124 ],
       ...,
       [-4.239068 ,  4.738016 ],
       [-4.248826 ,  4.7392206],
       [-4.2277913,  4.715423 ]], dtype=float32), label_ids=None, metrics={'test_runtime': 188.3229, 'test_samples_per_second': 79.289, 'test_steps_per_second': 4.96})

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.05,
    logging_steps=100,
)